# Uniform split sweep, fixed tuner-best hyperparameters + per-sample predictions

Companion to `ml_38_uniform_data_amount_sweep_keras_tuner.ipynb`, doing what `ml_37` does for the corner sweep.

No fresh tuner search here, we load the per-fraction best HPs ml_38 already found (from `data_amount_sweep_uniform_keras_tuner_seed0.csv`) and retrain one model per (fraction, seed) with those fixed HPs. Single seed, `SEEDS = (1,)`, on purpose different from ml_38s tuner seed 0 so this is a real retrain and not a repeat of the tuner-best run. Same uniform split (`SPLIT_SEED = 42`) as ml_35/ml_36/ml_38 so the slices and held-out rows are identical.

Per (fraction, seed):

1. rebuild the surrogate with that fractions tuner-best surrogate HP, train on the nested uniform subset
2. freeze it, rebuild + train the inverse with that fractions tuner-best inverse HP (same `qiskit_range_penalty` weight)

Outputs:

- `results/data_amount_sweep_uniform/data_amount_sweep_uniform_fixed_hp_seed{S}.csv` (ml_37 schema, one per seed)
- `results/data_amount_sweep_uniform/data_amount_sweep_uniform_fixed_hp_all_seeds_summary.csv` (folds in the ml_38 seed 0 rows, which used the same per-fraction HP)
- `model/uniform_data_amount_sweep_fixed_hp/fraction_*pct_seed{S}_{surrogate,combined,inverse}.keras`
- `results/data_amount_sweep_uniform/predictions_per_sample_uniform_fixed_hp_all_seeds.csv`, 20 per-sample predictions per slice in the standard ml_37 column format (`*_true` / `*_pred` / `*_pct_error` for the Hamiltonian targets, `*_true_um` / `*_pred_um` for geometry). The 20 targets are `test_idx[:20]`, same fixed test samples ml_36 exports for the EM simulator, so rows compare directly across notebooks.

100% still means 100% of the (uniformly sampled) non-held-out training pool, uniform val/test rows always excluded.


In [1]:
import os                                                                                                                                                                                              
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

In [2]:
from __future__ import annotations

import gc
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
## Use the async CUDA allocator to avoid fragmentation OOMs over the long
## multi-fraction training loop. Must be set before TensorFlow touches the GPU.
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.models import load_model


tf.keras.backend.set_floatx("float32")

## Let GPU memory grow on demand instead of pre-reserving it all, which also
## reduces fragmentation OOMs during the long run.
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass


## This notebook can be run either from the transmon experiment folder or from the
## repo root. The block below tries both so local paths do not need hard-coding.
HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon-cross metadata file. "
        "Run this notebook from the repo root or from the transmon experiment folder."
    )

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from parameters_surrogate_defined_loss import (  # noqa: E402
    EPOCHS,
    MODEL_DIR as PARAM_MODEL_DIR,
    SCALERS_DIR as PARAM_SCALERS_DIR,
    TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS,
)
from parameters_surrogate import (  # noqa: E402
    EPOCHS as SURROGATE_EPOCHS,
    TRAIN_BATCH_SIZE as SURROGATE_TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE as SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS as SURROGATE_TRAIN_LOSS,
)

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
## Per-fraction best hyperparameters are read from the completed ml_38 tuner sweep.
## only the seed varies in this notebook; each fraction's architecture/LR/etc. are
## frozen to its tuner-best values.
HP_SOURCE_CSV = EXPERIMENT_DIR / "results/data_amount_sweep_uniform/data_amount_sweep_uniform_keras_tuner_seed0.csv"
## one results CSV is written per seed (see out_path_for_seed); this is the across-seed roll-up.
COMBINED_SUMMARY_OUT_PATH = EXPERIMENT_DIR / "results/data_amount_sweep_uniform/data_amount_sweep_uniform_fixed_hp_all_seeds_summary.csv"
## long-format per-sample predictions (one row per seed/fraction/test sample).
PRED_OUT_PATH = EXPERIMENT_DIR / "results/data_amount_sweep_uniform/predictions_per_sample_uniform_fixed_hp_all_seeds.csv"

MODEL_DIR = Path(PARAM_MODEL_DIR)
SCALERS_DIR = Path(PARAM_SCALERS_DIR)
SWEEP_MODEL_DIR = MODEL_DIR / "uniform_data_amount_sweep_fixed_hp"
SWEEP_MODEL_DIR.mkdir(parents=True, exist_ok=True)

## sweep all ten training-pool fractions (10% ... 100%).
FRACTIONS = tuple(round(0.1 * i, 2) for i in range(1, 11))
## ml_38 ran its tuner search with seed 0; retrain with one fresh seed so this is a
## genuine retrain with the fixed HP rather than a repeat of the tuner-best run.
## add more seeds to the tuple later for a multi-seed study (per-seed CSVs + resume
## already support it).
TUNER_SEED = 0
SEEDS = (1,)

## Uniform random split across the whole parameter space (no held-out corner).
## must match ml_38 (and ml_35/ml_36) exactly so the training slices and held-out
## test rows are the ones the hyperparameters were tuned for.
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15
SPLIT_SEED = 42

## training-loop settings for the inverse (combined) model, shared with ml_21/ml_38.
INVERSE_EPOCHS = EPOCHS
INVERSE_BATCH_SIZE = TRAIN_BATCH_SIZE
INVERSE_EARLY_STOPPING_PATIENCE = TRAIN_EARLY_STOPPING_PATIENCE

## Run the training sweep. Set this to False if the per-seed CSVs already exist and
## you only want the summary/predictions cells.
RUN_SWEEP = True

## Per-sample predictions: cap saved test samples per (seed, fraction); None -> all.
## test_idx[:20] is the same fixed 20-target set ml_36 exports for EM simulator.
N_SAMPLES_PER_SLICE = 20
## also predict with ml_38's saved tuner-best models (seed 0) in the predictions
## cell. Off by default to keep this a single-seed run; if enabled, the ml_38
## .keras files must exist locally (models are git-ignored, so run on the GPU box).
INCLUDE_TUNER_SEED_MODELS = False

EPS = 1e-12


In [3]:
FRACTIONS = tuple(round(0.1 * i, 2) for i in range(1, 9))   # 10% ... 80%                                                                                                                              
N_SAMPLES_PER_SLICE = 10                                                                                                                                                                               
PRED_OUT_PATH = EXPERIMENT_DIR / "results/data_amount_sweep_uniform/predictions_per_sample_uniform_fixed_hp_10to80pct.csv"                                                                                                               


In [4]:
## data loading

@dataclass
class Scaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.min_) / self.range_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.range_ + self.min_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


## Load Hamiltonian targets and geometry values from the SQuADDS metadata
def load_arrays() -> tuple[np.ndarray, np.ndarray]:
    data = json.loads(METADATA_PATH.read_text())
    hamiltonian = []
    geometry = []

    for row in data:
        h = row["Hamiltonian_params"]
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]

        hamiltonian.append(
            [
                float(h["qubit_frequency_GHz"]),
                float(h["anharmonicity_MHz"]),
            ]
        )
        geometry.append(
            [
                parse_um(readout["claw_length"]),
                parse_um(readout["ground_spacing"]),
                parse_um(opts["cross_length"]),
            ]
        )

    return np.asarray(hamiltonian, dtype=np.float64), np.asarray(geometry, dtype=np.float64)


def choose_uniform_split(
    n_rows: int,
    test_fraction: float,
    val_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Draw test/validation/training-pool indices uniformly at random.

    Unlike the corner notebook, no geometry corner is held out. The test and
    validation rows are a uniform random sample from across the whole parameter
    space, and the rest becomes the training pool.
    """
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    rng = np.random.default_rng(seed)
    perm = rng.permutation(n_rows)

    test_idx = perm[:n_test]
    val_idx = perm[n_test:n_test + n_val]
    train_pool_idx = perm[n_test + n_val:]

    return train_pool_idx, val_idx, test_idx


## Return a uniform random ordering (local indices) of the training pool
## Taking the first ``n`` of this ordering gives nested uniform random subsets
def uniform_training_order(train_pool_idx: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed + 1000)
    return rng.permutation(len(train_pool_idx))


def scaler_from_artifacts(
    values: np.ndarray,
    columns: list[str],
    path_patterns: list[str],
    label: str,
) -> tuple[Scaler, list[str]]:
    """Load per-column MinMaxScaler ranges when present, otherwise fit on metadata."""
    mins = []
    maxs = []
    sources = []

    for i, col in enumerate(columns):
        loaded = None
        source = None
        for pattern in path_patterns:
            candidate = SCALERS_DIR / pattern.format(col=col)
            if candidate.exists():
                loaded = joblib.load(candidate)
                source = str(candidate)
                break

        if loaded is not None:
            mins.append(float(np.asarray(loaded.data_min_).reshape(-1)[0]))
            maxs.append(float(np.asarray(loaded.data_max_).reshape(-1)[0]))
            sources.append(source)
        else:
            mins.append(float(np.min(values[:, i])))
            maxs.append(float(np.max(values[:, i])))
            sources.append(f"metadata fallback: {label}.{col}")

    return Scaler(np.asarray(mins), np.asarray(maxs)), sources


In [5]:
## building the split and scalers

h_raw, geom_raw_um = load_arrays()
geom_raw_si = geom_raw_um * 1e-6

HAMILTONIAN_COLUMN_NAMES = (METADATA_DIR / "X_names").read_text().splitlines()
QISKIT_PARAM_NAMES = np.load(METADATA_DIR / "y_columns.npy", allow_pickle=True).astype(str).tolist()

train_pool_idx, val_idx, test_idx = choose_uniform_split(
    len(geom_raw_um),
    TEST_FRACTION,
    VAL_FRACTION,
    SPLIT_SEED,
)

## Make sure the held-out validation/test rows never leak into the training pool.
assert len(np.intersect1d(train_pool_idx, val_idx)) == 0
assert len(np.intersect1d(train_pool_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0

## Uniform random ordering of the training pool; nested subsets grow with fraction.
uniform_order = uniform_training_order(train_pool_idx, SPLIT_SEED)

## For the actual model inputs/outputs, use the saved scaler artifacts from the
## existing repo workflow. These are fit on the complete dataset, so the scaled
## [0, 1] range used by the range penalty is the total space spanned by the
## complete dataset. This also keeps the surrogate in its original scaled space.
h_model_scaler, h_scaler_sources = scaler_from_artifacts(
    h_raw,
    HAMILTONIAN_COLUMN_NAMES,
    ["scaler_X_{col}.save", "scaler_X_linear_{col}.save"],
    "Hamiltonian",
)
geom_inverse_scaler, geom_inverse_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_{col}_one_hot_encoding.save"],
    "inverse_qiskit",
)
geom_surrogate_scaler, geom_surrogate_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_linear_{col}.save", "scaler_y_{col}_one_hot_encoding.save"],
    "surrogate_qiskit",
)

h_model_scaled = h_model_scaler.transform(h_raw).astype("float32")
geom_inverse_scaled = geom_inverse_scaler.transform(geom_raw_si).astype("float32")
geom_surrogate_scaled = geom_surrogate_scaler.transform(geom_raw_si).astype("float32")

## Convert inverse output scaler space to surrogate input scaler space.
## surrogate_scaled = inverse_scaled * scale_a + scale_b
scale_a = (geom_inverse_scaler.range_ / geom_surrogate_scaler.range_).astype("float32")
scale_b = ((geom_inverse_scaler.min_ - geom_surrogate_scaler.min_) / geom_surrogate_scaler.range_).astype("float32")

print(f"Loaded {len(h_raw)} total samples")
print(f"Training pool: {len(train_pool_idx)}")
print(f"Validation (uniform): {len(val_idx)}")
print(f"Test (uniform): {len(test_idx)}")
print()
print("Split check passed:")
print("  train/val overlap:", len(np.intersect1d(train_pool_idx, val_idx)))
print("  train/test overlap:", len(np.intersect1d(train_pool_idx, test_idx)))
print("  val/test overlap:", len(np.intersect1d(val_idx, test_idx)))
print("  100% means all non-held-out training-pool samples, not all samples.")
print()
print("Using saved model-space scalers from the existing repo workflow.")
for name, lo, hi, source in zip(HAMILTONIAN_COLUMN_NAMES, h_model_scaler.min_, h_model_scaler.max_, h_scaler_sources):
    print(f"  Hamiltonian {name}: {lo:.6g} to {hi:.6g}  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_inverse_scaler.min_, geom_inverse_scaler.max_, geom_inverse_scaler_sources):
    print(f"  Inverse geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_surrogate_scaler.min_, geom_surrogate_scaler.max_, geom_surrogate_scaler_sources):
    print(f"  Surrogate geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")

if any(source.startswith("metadata fallback") for source in h_scaler_sources + geom_inverse_scaler_sources + geom_surrogate_scaler_sources):
    print()
    print("Note: at least one scaler artifact was not found, so metadata-derived min/max ranges were used for that column.")


Loaded 1934 total samples
Training pool: 1352
Validation (uniform): 291
Test (uniform): 291

Split check passed:
  train/val overlap: 0
  train/test overlap: 0
  val/test overlap: 0
  100% means all non-held-out training-pool samples, not all samples.

Using saved model-space scalers from the existing repo workflow.
  Hamiltonian qubit_frequency_GHz: 3.21853 to 7.12659  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_qubit_frequency_GHz.save)
  Hamiltonian anharmonicity_MHz: -525.818 to -88.9577  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_anharmonicity_MHz.save)
  Inverse geometry design_options.connection_pads.readout.claw_length: 7e-05 to 0.0004 SI units  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_y_design_options.connection_pads.readout.claw_length_one_hot_encoding.save)
  Inverse ge

In [6]:
## shared model pieces: the scaled-space conversion between the inverse output and
## the surrogate input, the out-of-range penalty, and the frozen-surrogate loader.
## Identical to ml_32/ml_35/ml_37 so the saved models stay interchangeable.


class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault("trainable", False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = {"scale_a": list(np.asarray(scale_a, dtype=float)), "scale_b": list(np.asarray(scale_b, dtype=float))}

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config


## penalize inverse predictions outside the scaled [0, 1] training range
def qiskit_range_penalty(y_true_dummy, y_pred):
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)


def load_frozen_surrogate(surrogate_model_path: Path):
    surrogate_model = load_model(surrogate_model_path, compile=False)
    surrogate_model.trainable = False
    for layer in surrogate_model.layers:
        layer.trainable = False
    return surrogate_model


def evaluate_percent_error(
    combined_model: Model,
    h_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict:
    h_pred_scaled, _ = combined_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)

    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def evaluate_surrogate_model(
    surrogate_model: Model,
    geom_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict:
    h_pred_scaled = surrogate_model.predict(np.asarray(geom_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)
    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def inverse_range_stats(inverse_model: Sequential, h_scaled_in: np.ndarray) -> dict:
    qiskit_scaled = inverse_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    below = np.maximum(-qiskit_scaled, 0.0)
    above = np.maximum(qiskit_scaled - 1.0, 0.0)
    violation = below + above
    return {
        "qiskit_scaled_min": float(np.min(qiskit_scaled)),
        "qiskit_scaled_max": float(np.max(qiskit_scaled)),
        "qiskit_range_violation_mean": float(np.mean(violation)),
        "qiskit_range_violation_max": float(np.max(violation)),
    }


In [7]:
## fixed per-fraction hyperparameter builders + trainers.
##
## Architecture mirrors make_surrogate_hypermodel / make_inverse_hypermodel in the
## cell above, but every hyperparameter is read from HP_SOURCE_CSV instead of being
## searched. set_random_seed(seed) before each build makes the (otherwise identical)
## models differ only by initialisation/shuffling.

from tensorflow.keras.layers import BatchNormalization

HP_SOURCE = pd.read_csv(HP_SOURCE_CSV)
_hp_missing = [f for f in FRACTIONS if not np.isclose(HP_SOURCE["fraction"].astype(float), float(f)).any()]
if _hp_missing:
    raise ValueError(
        f"HP_SOURCE_CSV {HP_SOURCE_CSV} is missing tuner-best rows for fractions {_hp_missing}. "
        "Finish ml_38_uniform_data_amount_sweep_keras_tuner.ipynb for those fractions first."
    )


def best_hp_for_fraction(fraction: float) -> pd.Series:
    mask = np.isclose(HP_SOURCE["fraction"].astype(float), float(fraction))
    return HP_SOURCE.loc[mask].iloc[0]


## surrogate with this fraction's tuner-best HP. Mirrors make_surrogate_hypermodel
def build_surrogate_fixed(input_dim: int, output_dim: int, row: pd.Series, seed: int) -> Sequential:
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)

    units = [int(u) for u in json.loads(row["surrogate_dense_units"])]
    dropout_rate = float(row["surrogate_dropout_rate"])
    l2_reg = float(row["surrogate_l2_reg"])
    lr_initial = float(row["surrogate_learning_rate"])
    use_batchnorm = bool(row["surrogate_use_batchnorm"])

    model = Sequential(name="retrained_surrogate")
    model.add(Input(shape=(input_dim,), name="input1"))
    for i, n_units in enumerate(units):
        model.add(Dense(n_units, name=f"fc{i}", kernel_initializer="he_normal",
                        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        if use_batchnorm:
            model.add(BatchNormalization(name=f"bn{i}"))
        model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
        model.add(Dropout(rate=dropout_rate, name=f"dropout{i}"))
    model.add(Dense(output_dim, name="output", kernel_initializer="he_normal"))
    model.compile(optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
                  loss=SURROGATE_TRAIN_LOSS, metrics=[SURROGATE_TRAIN_LOSS])
    return model


def build_inverse_fixed(h_dim: int, qiskit_dim: int, row: pd.Series, seed: int,
                        surrogate_model_path: Path) -> "tuple[Sequential, Model]":
    """Inverse + frozen surrogate with this fraction's tuner-best HP. Mirrors make_inverse_hypermodel."""
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(seed)

    units = [int(u) for u in json.loads(row["inverse_dense_units"])]
    dropout_rate = float(row["inverse_dropout_rate"])
    l2_reg = float(row["inverse_l2_reg"])
    lr_initial = float(row["inverse_learning_rate"])
    use_batchnorm = bool(row["inverse_use_batchnorm"])
    penalty_weight = float(row["range_penalty_weight"])

    surrogate_model = load_frozen_surrogate(surrogate_model_path)
    converter = ScalerConversionLayer(scale_a, scale_b, name="scaler_conversion")

    inverse_model = Sequential(name="inverse_model")
    inverse_model.add(Input(shape=(h_dim,), name="Hamiltonian_input"))
    for i, n_units in enumerate(units):
        inverse_model.add(Dense(n_units, name=f"fc{i}", kernel_initializer="he_normal",
                                kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
        if use_batchnorm:
            inverse_model.add(BatchNormalization(name=f"bn{i}"))
        inverse_model.add(LeakyReLU(negative_slope=0.01, name=f"leaky_relu{i}"))
        inverse_model.add(Dropout(rate=dropout_rate, name=f"dropout{i}"))
    inverse_model.add(Dense(qiskit_dim, name="qiskit_output", kernel_initializer="he_normal"))

    combined_input = Input(shape=(h_dim,), name="combined_input")
    predicted_qiskit = inverse_model(combined_input)
    predicted_qiskit_converted = converter(predicted_qiskit)
    reconstructed_hamiltonian = surrogate_model(predicted_qiskit_converted)
    combined_model = Model(
        inputs=combined_input,
        outputs=[reconstructed_hamiltonian, predicted_qiskit],
        name="combined_model",
    )
    combined_model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
        loss=[TRAIN_LOSS, qiskit_range_penalty],
        loss_weights=[1.0, penalty_weight],
    )
    return inverse_model, combined_model


def train_surrogate_fixed(model, geom_subset_scaled, h_subset_scaled, geom_val_scaled, h_val_scaled):
    early_stopping = EarlyStopping(monitor="val_loss", mode="min",
                                   patience=SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
                                   restore_best_weights=True, verbose=0)
    reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=max(10, SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE // 3),
                                  min_lr=1e-6, verbose=0)
    history = model.fit(
        np.asarray(geom_subset_scaled, dtype="float32"),
        np.asarray(h_subset_scaled, dtype="float32"),
        epochs=SURROGATE_EPOCHS, batch_size=SURROGATE_TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(geom_val_scaled, dtype="float32"),
                         np.asarray(h_val_scaled, dtype="float32")),
        callbacks=[early_stopping, reduce_lr], verbose=0,
    )
    val_hist = history.history.get("val_loss", [])
    best_val = float(np.min(val_hist)) if val_hist else float("nan")
    best_epoch = int(np.argmin(val_hist)) if val_hist else -1
    return best_val, best_epoch


def train_inverse_fixed(combined_model, h_subset_scaled, h_val_scaled):
    qiskit_dim = len(QISKIT_PARAM_NAMES)
    dummy_train = np.zeros((len(h_subset_scaled), qiskit_dim), dtype="float32")
    dummy_val = np.zeros((len(h_val_scaled), qiskit_dim), dtype="float32")
    early_stopping = EarlyStopping(monitor="val_loss", mode="min",
                                   patience=INVERSE_EARLY_STOPPING_PATIENCE,
                                   restore_best_weights=True, verbose=0)
    reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                  patience=max(10, INVERSE_EARLY_STOPPING_PATIENCE // 3),
                                  min_lr=1e-6, verbose=0)
    history = combined_model.fit(
        np.asarray(h_subset_scaled, dtype="float32"),
        [np.asarray(h_subset_scaled, dtype="float32"), dummy_train],
        epochs=INVERSE_EPOCHS, batch_size=INVERSE_BATCH_SIZE,
        validation_data=(np.asarray(h_val_scaled, dtype="float32"),
                         [np.asarray(h_val_scaled, dtype="float32"), dummy_val]),
        callbacks=[early_stopping, reduce_lr], verbose=0,
    )
    val_hist = history.history.get("val_loss", [])
    best_val = float(np.min(val_hist)) if val_hist else float("nan")
    best_epoch = int(np.argmin(val_hist)) if val_hist else -1
    return best_val, best_epoch


print("Loaded per-fraction best hyperparameters from:", HP_SOURCE_CSV)
print("Fractions available:", sorted(round(float(f), 2) for f in HP_SOURCE["fraction"].unique()))
print("Seeds to train:", SEEDS)


Loaded per-fraction best hyperparameters from: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_keras_tuner_seed0.csv
Fractions available: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
Seeds to train: (1,)


In [8]:
## fixed-HP retraining sweep: one model per (fraction, seed), one results CSV per seed.

def model_paths_for_fraction_seed(fraction: float, seed: int) -> tuple:
    pct = int(round(fraction * 100))
    stem = f"fraction_{pct:03d}pct_seed{seed}"
    return (
        SWEEP_MODEL_DIR / f"{stem}_surrogate.keras",
        SWEEP_MODEL_DIR / f"{stem}_combined.keras",
        SWEEP_MODEL_DIR / f"{stem}_inverse.keras",
    )


def out_path_for_seed(seed: int) -> Path:
    return EXPERIMENT_DIR / f"results/data_amount_sweep_uniform/data_amount_sweep_uniform_fixed_hp_seed{seed}.csv"


all_rows = []

if RUN_SWEEP:
    n_train_pool = len(train_pool_idx)
    print(f"Fixed-HP retraining over {len(FRACTIONS)} fractions x {len(SEEDS)} seeds.")
    print(f"Sweep model output dir: {SWEEP_MODEL_DIR}")

    for seed in SEEDS:
        seed_out_path = out_path_for_seed(seed)

        ## Resume support: keep rows already written for this seed so an interrupted
        ## run continues instead of clobbering finished fractions.
        if seed_out_path.exists():
            rows = pd.read_csv(seed_out_path).to_dict("records")
            done_pcts = {int(round(r["fraction"] * 100)) for r in rows}
            print(f"\nSeed {seed}: resuming from {seed_out_path} ({len(rows)} existing rows)")
        else:
            rows = []
            done_pcts = set()
            print(f"\nSeed {seed}: starting fresh -> {seed_out_path}")

        for fraction in FRACTIONS:
            n_subset = max(1, int(round(fraction * n_train_pool)))
            subset_local = uniform_order[:n_subset]
            subset_idx = train_pool_idx[subset_local]

            surrogate_model_path, combined_model_path, inverse_model_path = model_paths_for_fraction_seed(fraction, seed)

            if int(round(fraction * 100)) in done_pcts and combined_model_path.exists():
                print(f"  skip {fraction:.0%} seed {seed} (already complete)")
                continue

            row_hp = best_hp_for_fraction(fraction)
            print(f"  === seed {seed} | {fraction:.0%} of pool ({n_subset} samples) ===")

            ## forward surrogate (fixed HP)
            surrogate_model = build_surrogate_fixed(
                geom_surrogate_scaled.shape[1], h_model_scaled.shape[1], row_hp, seed,
            )
            surrogate_best_val_loss, surrogate_best_step = train_surrogate_fixed(
                surrogate_model,
                geom_surrogate_scaled[subset_idx], h_model_scaled[subset_idx],
                geom_surrogate_scaled[val_idx], h_model_scaled[val_idx],
            )
            surrogate_model.save(surrogate_model_path)

            surrogate_train_metrics = evaluate_surrogate_model(surrogate_model, geom_surrogate_scaled[subset_idx], h_raw[subset_idx])
            surrogate_val_metrics = evaluate_surrogate_model(surrogate_model, geom_surrogate_scaled[val_idx], h_raw[val_idx])
            surrogate_test_metrics = evaluate_surrogate_model(surrogate_model, geom_surrogate_scaled[test_idx], h_raw[test_idx])

            del surrogate_model
            tf.keras.backend.clear_session()
            gc.collect()

            ## inverse + frozen surrogate (fixed HP)
            inverse_model, combined_model = build_inverse_fixed(
                h_model_scaled.shape[1], len(QISKIT_PARAM_NAMES), row_hp, seed, surrogate_model_path,
            )
            inverse_best_val_loss, inverse_best_step = train_inverse_fixed(
                combined_model, h_model_scaled[subset_idx], h_model_scaled[val_idx],
            )
            combined_model.save(combined_model_path)
            inverse_model.save(inverse_model_path)

            train_metrics = evaluate_percent_error(combined_model, h_model_scaled[subset_idx], h_raw[subset_idx])
            val_metrics = evaluate_percent_error(combined_model, h_model_scaled[val_idx], h_raw[val_idx])
            test_metrics = evaluate_percent_error(combined_model, h_model_scaled[test_idx], h_raw[test_idx])
            test_range_stats = inverse_range_stats(inverse_model, h_model_scaled[test_idx])

            rows.append(
                {
                    "selection_method": "uniform_random_split_nested_random_subsets_fixed_per_fraction_hp",
                    "split_seed": SPLIT_SEED,
                    "fraction": fraction,
                    "training_percent": fraction * 100.0,
                    "n_samples": n_subset,
                    "seed": seed,
                    "hyperparameter_source": str(HP_SOURCE_CSV),
                    "surrogate_training_mode": "fixed_per_fraction_tuner_hp_retrained_per_seed",
                    "saved_surrogate_model_path": str(surrogate_model_path),
                    "saved_combined_model_path": str(combined_model_path),
                    "saved_inverse_model_path": str(inverse_model_path),
                    "surrogate_dense_units": row_hp["surrogate_dense_units"],
                    "inverse_dense_units": row_hp["inverse_dense_units"],
                    "surrogate_optimizer": "Adam",
                    "surrogate_learning_rate": float(row_hp["surrogate_learning_rate"]),
                    "surrogate_dropout_rate": float(row_hp["surrogate_dropout_rate"]),
                    "surrogate_l2_reg": float(row_hp["surrogate_l2_reg"]),
                    "surrogate_use_batchnorm": bool(row_hp["surrogate_use_batchnorm"]),
                    "surrogate_reconstruction_loss": SURROGATE_TRAIN_LOSS,
                    "surrogate_best_val_loss": surrogate_best_val_loss,
                    "surrogate_best_epoch": surrogate_best_step,
                    "surrogate_jit_compile": False,
                    "surrogate_batch_size": SURROGATE_TRAIN_BATCH_SIZE,
                    "surrogate_early_stopping_patience": SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
                    "inverse_optimizer": "Adam",
                    "inverse_learning_rate": float(row_hp["inverse_learning_rate"]),
                    "inverse_n_layers": int(row_hp["inverse_n_layers"]),
                    "inverse_dropout_rate": float(row_hp["inverse_dropout_rate"]),
                    "inverse_l2_reg": float(row_hp["inverse_l2_reg"]),
                    "inverse_use_batchnorm": bool(row_hp["inverse_use_batchnorm"]),
                    "inverse_reconstruction_loss": TRAIN_LOSS,
                    "range_penalty_weight": float(row_hp["range_penalty_weight"]),
                    "inverse_best_val_loss": inverse_best_val_loss,
                    "inverse_best_epoch": inverse_best_step,
                    "inverse_jit_compile": False,
                    "inverse_batch_size": INVERSE_BATCH_SIZE,
                    "inverse_early_stopping_patience": INVERSE_EARLY_STOPPING_PATIENCE,
                    **{f"surrogate_train_{key}": value for key, value in surrogate_train_metrics.items()},
                    **{f"surrogate_val_{key}": value for key, value in surrogate_val_metrics.items()},
                    **{f"surrogate_test_{key}": value for key, value in surrogate_test_metrics.items()},
                    **{f"train_{key}": value for key, value in train_metrics.items()},
                    **{f"val_{key}": value for key, value in val_metrics.items()},
                    **{f"test_{key}": value for key, value in test_metrics.items()},
                    **{f"test_{key}": value for key, value in test_range_stats.items()},
                }
            )

            ## write incrementally so a long sweep stays recoverable if interrupted.
            pd.DataFrame(rows).to_csv(seed_out_path, index=False)

            del inverse_model, combined_model
            tf.keras.backend.clear_session()
            gc.collect()

        pd.DataFrame(rows).to_csv(seed_out_path, index=False)
        print(f"  wrote {seed_out_path}")
        all_rows.extend(rows)

    out = pd.DataFrame(all_rows)
    print()
    print(f"Done. {len(SEEDS)} per-seed CSVs written; saved sweep models to {SWEEP_MODEL_DIR}")
else:
    ## read back all per-seed CSVs without retraining.
    frames = [pd.read_csv(out_path_for_seed(seed)) for seed in SEEDS if out_path_for_seed(seed).exists()]
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print("RUN_SWEEP is False; loaded existing per-seed CSVs.")

out.head()


Fixed-HP retraining over 8 fractions x 1 seeds.
Sweep model output dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/uniform_data_amount_sweep_fixed_hp

Seed 1: starting fresh -> /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_fixed_hp_seed1.csv
  === seed 1 | 10% of pool (135 samples) ===
  === seed 1 | 20% of pool (270 samples) ===
  === seed 1 | 30% of pool (406 samples) ===
  === seed 1 | 40% of pool (541 samples) ===
  === seed 1 | 50% of pool (676 samples) ===
  === seed 1 | 60% of pool (811 samples) ===
  === seed 1 | 70% of pool (946 samples) ===
  === seed 1 | 80% of pool (1082 samples) ===
  wrote /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_fixed_hp_seed1.csv

Done. 1 per-seed CSVs written; saved sweep models to /home/olivias/ML_qubit_design/experiments/model_predict_qubit_

,selection_method,split_seed,fraction,training_percent,n_samples,seed,hyperparameter_source,surrogate_training_mode,saved_surrogate_model_path,saved_combined_model_path,...,val_omega_q_mean_pct,val_alpha_mean_pct,val_mean_hamiltonian_pct,test_omega_q_mean_pct,test_alpha_mean_pct,test_mean_hamiltonian_pct,test_qiskit_scaled_min,test_qiskit_scaled_max,test_qiskit_range_violation_mean,test_qiskit_range_violation_max
0,uniform_random_split_nested_random_subsets_fix...,42,0.1,10.0,135,1,/home/olivias/ML_qubit_design/experiments/mode...,fixed_per_fraction_tuner_hp_retrained_per_seed,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.760556,1.003677,0.882117,0.721089,0.972629,0.846859,-0.254424,0.999768,0.016106,0.254424
1,uniform_random_split_nested_random_subsets_fix...,42,0.2,20.0,270,1,/home/olivias/ML_qubit_design/experiments/mode...,fixed_per_fraction_tuner_hp_retrained_per_seed,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,1.179753,2.411539,1.795646,1.768236,3.369810,2.569023,0.086623,10.366611,0.035784,9.366611
2,uniform_random_split_nested_random_subsets_fix...,42,0.3,30.0,406,1,/home/olivias/ML_qubit_design/experiments/mode...,fixed_per_fraction_tuner_hp_retrained_per_seed,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.224131,0.406242,0.315186,0.273184,0.470039,0.371612,0.153597,2.844297,0.032517,1.844297
3,uniform_random_split_nested_random_subsets_fix...,42,0.4,40.0,541,1,/home/olivias/ML_qubit_design/experiments/mode...,fixed_per_fraction_tuner_hp_retrained_per_seed,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.236804,0.585977,0.411390,0.256711,0.591431,0.424071,-0.495902,1.008279,0.000927,0.495902
4,uniform_random_split_nested_random_subsets_fix...,42,0.5,50.0,676,1,/home/olivias/ML_qubit_design/experiments/mode...,fixed_per_fraction_tuner_hp_retrained_per_seed,/home/olivias/ML_qubit_design/experiments/mode...,/home/olivias/ML_qubit_design/experiments/mode...,...,0.069172,0.267565,0.168369,0.084834,0.289244,0.187039,-0.103313,1.269411,0.014488,0.269411


In [9]:
## Across-seed summary: mean / std of the held-out test errors at each fraction.
## folds in the ml_38 seed-0 tuner rows, which used the identical per-fraction HP
## (they are where the HP came from), so the stats cover every trained seed.

SUMMARY_METRIC_COLS = [
    "training_percent", "n_samples", "seed",
    "train_mean_hamiltonian_pct", "val_mean_hamiltonian_pct", "test_mean_hamiltonian_pct",
]

frames = []
if len(out):
    frames.append(out)
## Fold in the ml_38 seed-0 tuner run so the stats cover it as well.
if HP_SOURCE_CSV.exists():
    frames.append(pd.read_csv(HP_SOURCE_CSV))

if frames:
    combined = pd.concat([f[[c for c in SUMMARY_METRIC_COLS if c in f.columns]] for f in frames], ignore_index=True)
    ## one row per (training_percent, seed); keep the first if anything duplicates.
    combined = combined.drop_duplicates(subset=["training_percent", "seed"], keep="first")
    summary = (
        combined.groupby(["training_percent", "n_samples"], as_index=False)
        .agg(
            n_seeds=("seed", "nunique"),
            train_mean=("train_mean_hamiltonian_pct", "mean"),
            train_std=("train_mean_hamiltonian_pct", "std"),
            val_mean=("val_mean_hamiltonian_pct", "mean"),
            val_std=("val_mean_hamiltonian_pct", "std"),
            test_mean=("test_mean_hamiltonian_pct", "mean"),
            test_std=("test_mean_hamiltonian_pct", "std"),
        )
        .sort_values("training_percent")
    )
    summary.to_csv(COMBINED_SUMMARY_OUT_PATH, index=False)
    print(f"wrote {COMBINED_SUMMARY_OUT_PATH}")
    print("Seeds per fraction:")
    print(summary[["training_percent", "n_seeds"]].to_string(index=False))
else:
    summary = pd.DataFrame()
    print("No rows available to summarize.")

summary


wrote /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_uniform_fixed_hp_all_seeds_summary.csv
Seeds per fraction:
 training_percent  n_seeds
             10.0        2
             20.0        2
             30.0        2
             40.0        2
             50.0        2
             60.0        2
             70.0        2
             80.0        2


,training_percent,n_samples,n_seeds,train_mean,train_std,val_mean,val_std,test_mean,test_std
0,10.0,135,2,0.813376,0.022275,0.855861,0.037131,0.845277,0.002237
1,20.0,270,2,1.299283,0.558337,1.345385,0.636765,2.080446,0.690953
2,30.0,406,2,0.335410,0.050072,0.348530,0.047155,0.409292,0.053288
3,40.0,541,2,0.342004,0.085186,0.344297,0.094885,0.358589,0.092605
4,50.0,676,2,0.194410,0.022877,0.188333,0.028234,0.203114,0.022733
5,60.0,811,2,0.478325,0.073454,0.429258,0.084633,0.502455,0.071307
6,70.0,946,2,0.939626,0.765637,0.890452,0.685005,0.948033,0.768065
7,80.0,1082,2,0.137080,0.011943,0.141928,0.008045,0.138615,0.001212


In [10]:
## Per-sample predictions on the held-out uniform test rows, all fractions x seeds.
## Reloads the already-trained combined models (no retraining) and writes long-format:
## one row per (seed, fraction, test_sample) in the standard ml_37 column format.
## save_idx = test_idx[:N_SAMPLES_PER_SLICE] is the same fixed 20-target set ml_36
## exports for EM simulator (same SPLIT_SEED), so the rows are directly comparable.

_custom = {"ScalerConversionLayer": ScalerConversionLayer, "qiskit_range_penalty": qiskit_range_penalty}


## combined model path for a (fraction, seed): this notebook's dir for the
## retrained seeds, HP_SOURCE's recorded ml_38 path for the tuner seed
def _combined_path(fraction: float, seed: int) -> Path:
    if seed == TUNER_SEED:
        row = best_hp_for_fraction(fraction)
        return Path(row["saved_combined_model_path"])
    return model_paths_for_fraction_seed(fraction, seed)[1]


seeds_all = tuple(SEEDS) + ((TUNER_SEED,) if INCLUDE_TUNER_SEED_MODELS else ())
## keep only the first N_SAMPLES_PER_SLICE test samples (same subset for every seed/fraction).
save_idx = test_idx if N_SAMPLES_PER_SLICE is None else test_idx[:N_SAMPLES_PER_SLICE]
h_true_test = h_raw[save_idx]                 # (n_save, n_h)  true Hamiltonian params
geom_true_um_test = geom_raw_um[save_idx]     # (n_save, n_q)  true qiskit geometry [um]

pred_rows = []
for seed in seeds_all:
    for fraction in FRACTIONS:
        cpath = _combined_path(fraction, seed)
        if not Path(cpath).exists():
            print(f"  MISSING combined model, skipping: seed {seed} {fraction:.0%} -> {cpath}")
            continue

        combined_model = load_model(cpath, compile=False, custom_objects=_custom)
        h_pred_scaled, qiskit_pred_scaled = combined_model.predict(
            np.asarray(h_model_scaled[save_idx], dtype="float32"), verbose=0
        )
        h_pred = h_model_scaler.inverse_transform(h_pred_scaled)                       # omega_q, alpha
        geom_pred_um = geom_inverse_scaler.inverse_transform(qiskit_pred_scaled) * 1e6  # SI -> um
        pct = 100.0 * np.abs(h_pred - h_true_test) / np.maximum(np.abs(h_true_test), EPS)

        for j, global_idx in enumerate(save_idx):
            rec = {
                "seed": seed,
                "fraction": fraction,
                "training_percent": fraction * 100.0,
                "test_sample_index": int(global_idx),   # index into the full dataset
                "test_sample_rank": j,                   # 0..n_save-1 within the test split
                "saved_combined_model_path": str(cpath),
            }
            for k, name in enumerate(HAMILTONIAN_COLUMN_NAMES):
                rec[f"{name}_true"] = float(h_true_test[j, k])
                rec[f"{name}_pred"] = float(h_pred[j, k])
                rec[f"{name}_pct_error"] = float(pct[j, k])
            for k, name in enumerate(QISKIT_PARAM_NAMES):
                rec[f"{name}_true_um"] = float(geom_true_um_test[j, k])
                rec[f"{name}_pred_um"] = float(geom_pred_um[j, k])
            pred_rows.append(rec)

        del combined_model
        tf.keras.backend.clear_session()
        gc.collect()
        print(f"  seed {seed} | {fraction:.0%}: {len(save_idx)} samples")

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(PRED_OUT_PATH, index=False)
print(f"\nWrote {len(pred_df)} rows "
      f"({pred_df['seed'].nunique()} seeds x {pred_df['fraction'].nunique()} fractions "
      f"x {len(save_idx)} test samples) -> {PRED_OUT_PATH}")
pred_df.head()


  seed 1 | 10%: 10 samples
  seed 1 | 20%: 10 samples
  seed 1 | 30%: 10 samples
  seed 1 | 40%: 10 samples
  seed 1 | 50%: 10 samples
  seed 1 | 60%: 10 samples
  seed 1 | 70%: 10 samples
  seed 1 | 80%: 10 samples

Wrote 80 rows (1 seeds x 8 fractions x 10 test samples) -> /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/predictions_per_sample_uniform_fixed_hp_10to80pct.csv


,seed,fraction,training_percent,test_sample_index,test_sample_rank,saved_combined_model_path,qubit_frequency_GHz_true,qubit_frequency_GHz_pred,qubit_frequency_GHz_pct_error,anharmonicity_MHz_true,anharmonicity_MHz_pred,anharmonicity_MHz_pct_error,design_options.connection_pads.readout.claw_length_true_um,design_options.connection_pads.readout.claw_length_pred_um,design_options.connection_pads.readout.ground_spacing_true_um,design_options.connection_pads.readout.ground_spacing_pred_um,design_options.cross_length_true_um,design_options.cross_length_pred_um
0,1,0.1,10.0,1746,0,/home/olivias/ML_qubit_design/experiments/mode...,4.028765,4.014061,0.364970,-144.058729,-145.083711,0.711503,90.0,55.073903,4.1,9.120889,280.0,275.092388
1,1,0.1,10.0,343,1,/home/olivias/ML_qubit_design/experiments/mode...,3.956432,3.942273,0.357887,-138.512559,-139.482608,0.700333,270.0,57.542053,4.1,9.201471,290.0,286.437027
2,1,0.1,10.0,1768,2,/home/olivias/ML_qubit_design/experiments/mode...,4.104551,4.089914,0.356604,-150.006988,-151.002098,0.663376,140.0,52.466232,4.1,9.035679,270.0,263.104634
3,1,0.1,10.0,415,3,/home/olivias/ML_qubit_design/experiments/mode...,5.731559,5.666857,1.128874,-314.818821,-312.313946,0.795656,70.0,15.211008,4.1,8.364253,140.0,137.060311
4,1,0.1,10.0,28,4,/home/olivias/ML_qubit_design/experiments/mode...,6.190370,6.078859,1.801367,-375.826539,-364.437374,3.030431,90.0,5.770011,4.1,8.428425,120.0,117.987936


In [13]:
ANSYS_EXPORT_DIR = EXPERIMENT_DIR / "results" / "validation" / "data_amount_sweep_uniform_fixed_hp_em_sim"
ANSYS_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DETAIL_OUT_PATH = ANSYS_EXPORT_DIR / "uniform_fixed_hp_test_predictions_10to80pct.csv"
ANSYS_INPUT_OUT_PATH = ANSYS_EXPORT_DIR / "uniform_fixed_hp_test_em_sim_input_10to80pct.csv"

n_train_pool = len(train_pool_idx)

exp = pred_df.copy()
exp["ansys_job_id"] = [
    f"pct{int(round(tp)):03d}_seed{int(s)}_test{int(r):03d}"
    for tp, s, r in zip(exp["training_percent"], exp["seed"], exp["test_sample_rank"])
]
exp["target_set"] = f"uniform_test_{len(save_idx)}_samples"
exp["split_seed"] = SPLIT_SEED
exp["n_samples"] = [max(1, int(round(f * n_train_pool))) for f in exp["fraction"]]
exp["test_sample_number"] = exp["test_sample_rank"]
exp["metadata_index"] = exp["test_sample_index"]
exp["selection_method"] = "uniform_random_split_nested_random_subsets_fixed_per_fraction_hp"

## EM simulator-input contract columns (ref_/pred_, SI units), aliased from the standard-format columns.
exp["ref_qubit_frequency_GHz"] = exp["qubit_frequency_GHz_true"]
exp["ref_anharmonicity_MHz"] = exp["anharmonicity_MHz_true"]
exp["pred_qubit_frequency_GHz"] = exp["qubit_frequency_GHz_pred"]
exp["pred_anharmonicity_MHz"] = exp["anharmonicity_MHz_pred"]
for _short in ("connection_pads.readout.claw_length", "connection_pads.readout.ground_spacing", "cross_length"):
    exp[f"ref_{_short}"] = exp[f"design_options.{_short}_true_um"] * 1e-6
    exp[f"pred_{_short}"] = exp[f"design_options.{_short}_pred_um"] * 1e-6

exp["mean_hamiltonian_pct_error"] = exp[["qubit_frequency_GHz_pct_error", "anharmonicity_MHz_pct_error"]].mean(axis=1)

ANSYS_RESULT_COLUMNS = [
    "ansys_qubit_frequency_GHz",
    "ansys_anharmonicity_MHz",
    "ansys_frequency_pct_error",
    "ansys_anharmonicity_pct_error",
    "ansys_mean_hamiltonian_pct_error",
]
for _col in ANSYS_RESULT_COLUMNS:
    exp[_col] = np.nan

front_cols = [
    "ansys_job_id", "target_set", "fraction", "training_percent", "seed", "split_seed",
    "n_samples", "test_sample_number", "metadata_index", "selection_method",
]
ANSYS_INPUT_COLUMNS = [
    "ref_qubit_frequency_GHz",
    "ref_anharmonicity_MHz",
    "ref_connection_pads.readout.claw_length",
    "ref_connection_pads.readout.ground_spacing",
    "ref_cross_length",
    "pred_qubit_frequency_GHz",
    "pred_anharmonicity_MHz",
    "pred_connection_pads.readout.claw_length",
    "pred_connection_pads.readout.ground_spacing",
    "pred_cross_length",
]

exp.to_csv(DETAIL_OUT_PATH, index=False)
exp[front_cols + ANSYS_INPUT_COLUMNS + ANSYS_RESULT_COLUMNS].to_csv(ANSYS_INPUT_OUT_PATH, index=False)

print(f"Wrote full detail CSV  -> {DETAIL_OUT_PATH}")
print(f"Wrote EM simulator input CSV  -> {ANSYS_INPUT_OUT_PATH}")
print("Rows to simulate:", len(exp), f"({exp['fraction'].nunique()} fractions x {len(save_idx)} samples)")


Wrote full detail CSV  -> /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/validation/data_amount_sweep_uniform_fixed_hp_em_sim/uniform_fixed_hp_test_predictions_10to80pct.csv
Wrote EM simulator input CSV  -> /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/results/validation/data_amount_sweep_uniform_fixed_hp_em_sim/uniform_fixed_hp_test_em_sim_input_10to80pct.csv
Rows to simulate: 80 (8 fractions x 10 samples)
